# 🎬 Chat with Video using Qwen3 Omni on Fireworks AI

This cookbook demonstrates how to use **Qwen3 Omni**, a powerful multimodal model, to analyze and have conversations about video content. You'll learn how to:

1. Download and prepare video data
2. Set up an on-demand deployment of [Qwen3 Omni Instruct](https://fireworks.ai/models/fireworks/qwen3-omni-30b-a3b-instruct) on Fireworks AI
3. Preprocess video and audio for optimal model performance
4. Format and send requests to the model
5. Have interactive conversations about video content

By the end, you'll be able to ask questions about videos and receive intelligent, context-aware responses.

---
## 1. Background: What is Qwen3 Omni?

**Qwen3 Omni** (`qwen3-omni-30b-a3b-instruct`) is a state-of-the-art multimodal model that can process video frames, audio, and text in a single request. This enables powerful capabilities like:

- **Video Captioning**: Describe what's happening in a video
- **Scene Analysis**: Identify objects, people, and actions
- **Audio Understanding**: Transcribe speech, identify music, and describe sounds
- **Multimodal Q&A**: Answer questions that require understanding both visual and audio content

While most vision-language models treat video as a sequence of image frames, Qwen3 Omni accepts video and audio directly; this enables it to provide a more complete understanding of multimedia content by jointly reasoning across both audio and visual modalities.

In this cookbook, we'll use public domain videos from the Internet Archive to demonstrate these capabilities.

---
## 2. Setup: Install Dependencies

We'll need a few Python packages, `ffmpeg` for video preprocessing, and `firectl` to programmatically spin up deployments on Fireworks.

In [ ]:
# Install required packages
%pip install -q requests

# Install ffmpeg (required for video preprocessing)
!apt-get update -qq && apt-get install -qq ffmpeg

print("✅ Dependencies installed successfully!")

In [ ]:
import os
import subprocess
import tempfile
import base64
import requests
import urllib.request
from pathlib import Path
from IPython.display import Video, display, HTML

print("✅ Imports successful!")

### Configure Your API Key

You'll need a Fireworks AI API key. Get one at [fireworks.ai](https://fireworks.ai) and set it below.

In [ ]:
# Set your Fireworks API key using Colab secrets
from google.colab import userdata
os.environ["FIREWORKS_API_KEY"] = userdata.get("FIREWORKS_API_KEY")

# Verify API key is set
if os.environ.get("FIREWORKS_API_KEY"):
    print("✅ API key configured!")
else:
    print("❌ Please set your FIREWORKS_API_KEY")

---
## 3. Download Demo Videos

We'll use four public domain videos from the Internet Archive. These include vintage commercials, NASA footage, and newsreels; each have distinct visual and audio content that showcases the model's capabilities.

In [ ]:
# Define our demo videos from the Internet Archive
DEMO_VIDEOS = [
    {
        "id": "aurora_drag_race",
        "title": "Aurora Stunt and Drag Race Set Commercial",
        "description": "Television commercial for Aurora Stunt and Drag Race toy cars.",
        "url": "https://archive.org/download/aurora_drag_race_set/aurora_drag_race_set_512kb.mp4"
    },
    {
        "id": "cheerios_1960",
        "title": "Cheerios Space Age Commercial",
        "description": "Cereal commercial targeted at children of the space age.",
        "url": "https://archive.org/download/Cheerios1960/Cheerios1960_512kb.mp4"
    },
    {
        "id": "apollo_13_nixon",
        "title": "Apollo 13: Nixon Commends the Crew",
        "description": "Nixon commends the crew of Apollo 13 after their safe return.",
        "url": "https://archive.org/download/NIX-LV-1998-00042/LV-1998-00042_512kb.mp4"
    },
    {
        "id": "radio_transmitter_1950",
        "title": "Largest Radio Transmitter Dedicated",
        "description": "U.S. Navy opens 1 megawatt transmitter 'Radio Jim Creek' in Washington.",
        "url": "https://archive.org/download/LargestR1950/LargestR1950_512kb.mp4"
    }
]

print(f"📹 {len(DEMO_VIDEOS)} demo videos configured")

In [ ]:
def download_video(video_info: dict, output_dir: str = "./videos") -> str:
    """
    Download a video from URL and save locally.
    
    Args:
        video_info: Dictionary with 'id', 'title', and 'url' keys
        output_dir: Directory to save videos
    
    Returns:
        Path to the downloaded video file
    """
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{video_info['id']}.mp4")
    
    if os.path.exists(output_path):
        print(f"  ✓ Already downloaded: {video_info['title']}")
        return output_path
    
    print(f"  ↓ Downloading: {video_info['title']}...")
    urllib.request.urlretrieve(video_info['url'], output_path)
    print(f"  ✓ Saved to: {output_path}")
    
    return output_path


# Download all videos
print("📥 Downloading demo videos...\n")
video_paths = {}

for video in DEMO_VIDEOS:
    path = download_video(video)
    video_paths[video['id']] = {
        'path': path,
        'title': video['title'],
        'description': video['description']
    }

print(f"\n✅ Downloaded {len(video_paths)} videos!")

### Preview the Videos

Let's take a look at one of the videos we'll be working with.

In [ ]:
# Preview the first video
first_video = list(video_paths.values())[0]
print(f"🎬 {first_video['title']}")
print(f"   {first_video['description']}\n")

display(Video(first_video['path'], width=480))

---
## 4. Create a Fireworks On-Demand Deployment

Qwen3 Omni requires a **dedicated deployment** on Firewors since it is not currently available on serverless. Below, we will show you how to create a minimal Qwen3 Omni deployment using a predefined deployment shape.

### Understanding Deployment Shapes

Fireworks offers pre-configured **[Deployment Shapes](https://fireworks.ai/blog/deployment-shapes)** that make it easy to create optimized deployments. Instead of manually configuring quantization levels, GPU choices, and other settings, you can select a pre-configured shape, thus simplifying the setup.

For this cookbook, we'll use the **minimal** shape, which is ideal for experimentation and testing due to its lower cost.

### Create the Deployment

Use the Fireworks CLI (`firectl`) to create your deployment. Run this command in your terminal:

```bash
firectl create deployment accounts/fireworks/models/qwen3-omni-30b-a3b-instruct \
  --deployment-shape accounts/fireworks/deploymentShapes/qwen3-omni-30b-a3b-instruct-minimal
```

This will output a deployment ID. The deployment may take a few minutes to become ready. You can check its status with:

```bash
firectl list deployments
```

Once ready, configure your deployment details in the cell below.

In [ ]:
# Configure your deployment details
# Replace these with your actual account ID and deployment ID

FIREWORKS_ACCOUNT_ID = "your-account-id"  # e.g., "fireworks" or your account name
DEPLOYMENT_ID = "your-deployment-id"      # The deployment ID from firectl output

# Construct the full model string for API calls
MODEL_STRING = f"accounts/{FIREWORKS_ACCOUNT_ID}/models/qwen3-omni-30b-a3b-instruct#accounts/{FIREWORKS_ACCOUNT_ID}/deployments/{DEPLOYMENT_ID}"

print(f"📦 Model string configured:")
print(f"   {MODEL_STRING}")

---
## 5. Preprocess Video for the Model

Video models perform best with preprocessed inputs that balance quality and efficiency. The key optimizations:

1. **Reduce frame rate**: Extract 1 frame per second (the model doesn't need 30+ FPS)
2. **Downscale resolution**: 360p provides good quality with minimal tokens
3. **Extract audio separately**: Opus codec at 24kbps offers excellent compression
4. **Limit duration**: Cap at 60 seconds for consistent performance

We use `ffmpeg` to perform these optimizations. Here are the key parameters:

**Video preprocessing:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `-t 60` | 60 seconds | Limit video duration |
| `fps=1` | 1 FPS | Extract 1 frame per second |
| `scale=-1:360` | 360p | Downscale to 360p height |
| `-an` | — | Remove audio track |

**Audio preprocessing:**
| Parameter | Value | Description |
|-----------|-------|-------------|
| `-t 60` | 60 seconds | Limit audio duration |
| `-c:a libopus` | Opus | Use Opus codec |
| `-b:a 24k` | 24 kbps | Audio bitrate |
| `-ar 16000` | 16 kHz | Sample rate |
| `-ac 1` | Mono | Single audio channel |

In [ ]:
def preprocess_video(video_path: str) -> tuple[str, str]:
    """
    Preprocess video for optimal model input.
    
    This function:
    1. Extracts video at 1 FPS, 360p resolution
    2. Extracts audio as Opus/OGG at 24kbps, 16kHz, mono
    3. Limits both to 60 seconds maximum
    4. Returns base64-encoded strings for API transmission
    
    Args:
        video_path: Path to the input video file
    
    Returns:
        Tuple of (video_base64, audio_base64)
    """
    # Create temporary files for processed outputs
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp_video:
        processed_video_path = tmp_video.name
    with tempfile.NamedTemporaryFile(suffix=".ogg", delete=False) as tmp_audio:
        audio_path = tmp_audio.name
    
    try:
        # Process video: 1 FPS, 360p, max 60 seconds, no audio
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-t", "60",                    # Limit to 60 seconds
            "-vf", "fps=1,scale=-1:360",   # 1 FPS, 360p height
            "-c:v", "libx264",             # H.264 codec
            "-preset", "fast",
            "-an",                         # Remove audio
            processed_video_path
        ], check=True, capture_output=True)
        
        # Extract audio: Opus/OGG, mono, 16kHz, 24kbps
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path,
            "-t", "60",                    # Limit to 60 seconds
            "-vn",                         # Remove video
            "-c:a", "libopus",             # Opus codec
            "-b:a", "24k",                 # 24 kbps bitrate
            "-ar", "16000",                # 16 kHz sample rate
            "-ac", "1",                    # Mono audio
            audio_path
        ], check=True, capture_output=True)
        
        # Read and encode as base64
        with open(processed_video_path, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode("utf-8")
        
        with open(audio_path, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")
        
        return video_b64, audio_b64
    
    finally:
        # Clean up temporary files
        if os.path.exists(processed_video_path):
            os.unlink(processed_video_path)
        if os.path.exists(audio_path):
            os.unlink(audio_path)


print("✅ Preprocessing function defined!")

In [ ]:
# Preprocess all videos (this may take a minute)
print("🔄 Preprocessing videos...\n")

preprocessed_videos = {}

for video_id, video_info in video_paths.items():
    print(f"  Processing: {video_info['title']}...")
    video_b64, audio_b64 = preprocess_video(video_info['path'])
    
    preprocessed_videos[video_id] = {
        'title': video_info['title'],
        'description': video_info['description'],
        'video_b64': video_b64,
        'audio_b64': audio_b64,
        'original_path': video_info['path']
    }
    
    # Show size reduction
    original_size = os.path.getsize(video_info['path']) / 1024  # KB
    processed_size = (len(video_b64) + len(audio_b64)) * 3 / 4 / 1024  # Approx decoded KB
    print(f"    Original: {original_size:.1f} KB → Processed: {processed_size:.1f} KB")

print(f"\n✅ Preprocessed {len(preprocessed_videos)} videos!")

---
## 6. Sending Requests to Qwen3 Omni

Now we'll create a function to send requests to the model. The API uses the standard Chat Completions format, with video and audio provided as base64-encoded data URLs.

### Request Format

The message content is an array that can include multiple content types:

- `video_url`: Base64-encoded video as a data URL (`data:video/mp4;base64,...`)
- `audio_url`: Base64-encoded audio as a data URL (`data:audio/ogg;base64,...`)
- `text`: Your question or prompt

```python
payload = {
    "model": "accounts/<ACCOUNT>/models/qwen3-omni-30b-a3b-instruct#accounts/<ACCOUNT>/deployments/<DEPLOYMENT>",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": "data:video/mp4;base64,..."}},
                {"type": "audio_url", "audio_url": {"url": "data:audio/ogg;base64,..."}},
                {"type": "text", "text": "What is happening in this video?"}
            ]
        }
    ],
    "max_tokens": 1000,
    "temperature": 0.3
}
```

In [ ]:
def chat_with_video(video_b64: str, audio_b64: str, question: str) -> str:
    """
    Send a question about a video to Qwen3 Omni.
    
    Args:
        video_b64: Base64-encoded preprocessed video
        audio_b64: Base64-encoded preprocessed audio
        question: Your question about the video
    
    Returns:
        The model's response as a string
    """
    url = "https://api.fireworks.ai/inference/v1/chat/completions"
    
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}",
    }
    
    payload = {
        "model": MODEL_STRING,
        "max_tokens": 1000,
        "temperature": 0.3,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "video_url",
                        "video_url": {"url": f"data:video/mp4;base64,{video_b64}"}
                    },
                    {
                        "type": "audio_url",
                        "audio_url": {"url": f"data:audio/ogg;base64,{audio_b64}"}
                    },
                    {
                        "type": "text",
                        "text": question
                    }
                ],
            }
        ],
    }
    
    response = requests.post(url, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    
    return response.json()["choices"][0]["message"]["content"]


print("✅ Chat function defined!")

---
## 7. Interactive Video Chat

Now for the fun part! Let's have conversations with our videos. We'll ask questions about what's happening visually, what sounds are playing, and what's being said.

### Video 1: Aurora Drag Race Commercial

A vintage TV commercial for toy drag race cars. Let's see what the model can tell us about it.

In [ ]:
# Display the video
video_1 = preprocessed_videos['aurora_drag_race']
print(f"🎬 {video_1['title']}")
print(f"   {video_1['description']}\n")

display(Video(video_1['original_path'], width=480))

In [ ]:
# Ask about what's happening
question = "What is happening in this video? Describe the main action and any text you see."

print(f"❓ Question: {question}\n")
response = chat_with_video(video_1['video_b64'], video_1['audio_b64'], question)
print(f"💬 Response:\n{response}")

In [ ]:
# Ask about the audio
question = "What kind of music is playing in this commercial? Describe the audio and any voice-over."

print(f"❓ Question: {question}\n")
response = chat_with_video(video_1['video_b64'], video_1['audio_b64'], question)
print(f"💬 Response:\n{response}")

### Video 2: Cheerios Space Age Commercial

A 1960s cereal commercial with a space theme—perfect for testing the model's understanding of vintage advertising.

In [ ]:
# Display the video
video_2 = preprocessed_videos['cheerios_1960']
print(f"🎬 {video_2['title']}")
print(f"   {video_2['description']}\n")

display(Video(video_2['original_path'], width=480))

In [ ]:
# Ask about the product
question = "What product is being advertised? What claims are made about it?"

print(f"❓ Question: {question}\n")
response = chat_with_video(video_2['video_b64'], video_2['audio_b64'], question)
print(f"💬 Response:\n{response}")

In [ ]:
# Ask about the era
question = "Based on the visual style and content, what decade do you think this commercial is from? What clues tell you this?"

print(f"❓ Question: {question}\n")
response = chat_with_video(video_2['video_b64'], video_2['audio_b64'], question)
print(f"💬 Response:\n{response}")

### Video 3: Apollo 13 – Nixon Commends the Crew

Historic NASA footage of President Nixon honoring the Apollo 13 astronauts. A great test for speech transcription and historical context.

In [ ]:
# Display the video
video_3 = preprocessed_videos['apollo_13_nixon']
print(f"🎬 {video_3['title']}")
print(f"   {video_3['description']}\n")

display(Video(video_3['original_path'], width=480))

In [ ]:
# Ask about what's being said
question = "What is being said in this video? Summarize the main message."

print(f"❓ Question: {question}\n")
response = chat_with_video(video_3['video_b64'], video_3['audio_b64'], question)
print(f"💬 Response:\n{response}")

In [ ]:
# Ask about the people
question = "Who are the people in this video? Describe what you see them doing."

print(f"❓ Question: {question}\n")
response = chat_with_video(video_3['video_b64'], video_3['audio_b64'], question)
print(f"💬 Response:\n{response}")

### Video 4: Largest Radio Transmitter Dedicated

A 1950 newsreel about the U.S. Navy's powerful new radio transmitter. Tests the model's ability to understand technical content.

In [ ]:
# Display the video
video_4 = preprocessed_videos['radio_transmitter_1950']
print(f"🎬 {video_4['title']}")
print(f"   {video_4['description']}\n")

display(Video(video_4['original_path'], width=480))

In [ ]:
# Ask about the technical content
question = "What technology is being shown in this video? What makes it significant?"

print(f"❓ Question: {question}\n")
response = chat_with_video(video_4['video_b64'], video_4['audio_b64'], question)
print(f"💬 Response:\n{response}")

In [ ]:
# Ask about the narration style
question = "Describe the narration style and tone of this video. How does it compare to modern news coverage?"

print(f"❓ Question: {question}\n")
response = chat_with_video(video_4['video_b64'], video_4['audio_b64'], question)
print(f"💬 Response:\n{response}")

### Try Your Own Questions

Use the cell below to ask any question about any of the videos!

In [ ]:
# Choose a video and ask your own question
# Options: 'aurora_drag_race', 'cheerios_1960', 'apollo_13_nixon', 'radio_transmitter_1950'

selected_video_id = 'cheerios_1960'  # <-- Change this to select a different video
your_question = "What colors are most prominent in this video?"  # <-- Change this to your question

# Get the video
selected = preprocessed_videos[selected_video_id]
print(f"🎬 Video: {selected['title']}\n")

# Display and ask
display(Video(selected['original_path'], width=400))
print(f"\n❓ Question: {your_question}\n")

response = chat_with_video(selected['video_b64'], selected['audio_b64'], your_question)
print(f"💬 Response:\n{response}")

---
## 8. Conclusion

In this cookbook, we demonstrated how to use **Qwen3 Omni** on Fireworks AI to have intelligent conversations about video content. We covered:

1. **Video Preprocessing**: Using `ffmpeg` to optimize videos for the model (1 FPS, 360p, Opus audio)
2. **Deployment Setup**: Creating an on-demand deployment with deployment shapes for optimized performance
3. **API Integration**: Sending multimodal requests with video, audio, and text
4. **Interactive Chat**: Asking diverse questions about visual content, audio, and context

### Key Takeaways

- **Preprocessing matters**: Reducing frame rate and resolution significantly decreases latency and cost without sacrificing quality
- **Multimodal understanding**: The model can analyze both visual and audio content, enabling rich Q&A experiences
- **Deployment shapes simplify setup**: Pre-configured shapes let you quickly deploy optimized infrastructure

### Next Steps

- Try with your own videos—the same preprocessing approach works for any content
- Explore different question types: scene analysis, transcription, summarization, and more
- Build a production application using the patterns demonstrated here

### Resources

- [Fireworks AI Documentation](https://docs.fireworks.ai)
- [Deployment Shapes Blog](https://fireworks.ai/blog/deployment-shapes)
- [Qwen3 Omni Model Card](https://fireworks.ai/models/qwen3-omni-30b-a3b-instruct)

Happy building! 🚀

---

*This cookbook was created for educational purposes. All videos used are public domain content from the Internet Archive.*